# Utility Functions for Use in Other Python Scripts

## To use any of the functions in this notebool add this line when importing libraries:
%run utilities.ipynb

In [1]:
# https://stackoverflow.com/questions/44116194/import-a-function-from-another-ipynb-file
# !pip install ipynb

In [2]:
# time difference function
def timediff(start, end, decimals = 1):
    import datetime, time
    
    if   int((end - start) / 3600) > 0: # non-zero hours
        return str(int(  (end - start) / 3600)) + 'hr '  + str(int(  (end - start) / 60)          ) + 'min '\
         +                                                 str(round((end - start) % 60, decimals)) + 'sec'
    elif int((end - start) / 60)   > 0: # non-zero hours and minutes
        return str(int(  (end - start) / 60))   + 'min ' + str(round((end - start) % 60, decimals)) + 'sec'
    else:
        return                                             str(round((end - start) % 60, decimals)) + 'sec'

In [3]:
# function to convert .xls to .xlsx using win32 given a file path to the xls file
def xlsToXlsx(filepathInclXls):
    import win32com.client as win32 # library to convert xls to xlsx
    excel               = win32.gencache.EnsureDispatch('Excel.Application')
    excel.DisplayAlerts = False # suppress the warning dialogue
    wb = excel.Workbooks.Open(filepathInclXls)
    wb.SaveAs(filepathInclXls + 'x', FileFormat = 51)     # FileFormat = 51 (56) for .xlsx (.xlx) extension
    wb.Close()
    excel.DisplayAlerts = True  # unsuppress Excel warning dialogue
    return print(' ', filepathInclXls + 'x')

In [4]:
# eagle report types, their short codes, and their URLs
eagle_root         = r'https://eagleportal.prescient.co.za/Queries/Query.aspx?rpt='
report_types_dict  = {'r28i': ['Reg 28 Report - Incl Effective Exposure', eagle_root + 'Reg28withExposure' ],
                      'parn': ['Portfolio Analytics Report - New'       , eagle_root + 'PortfolioAnalytics'],
                      'derv': ['Derivative Exposure'                    , eagle_root + 'DerivativeExposure'],
                      'trad': ['Trades Report'                          , eagle_root + 'TRANSACTION'       ],
                      'scty': ['Security Cross Reference'               , eagle_root + 'SecurityCrossRef'  ],
                      'dflw': ['Daily Flows'                            , eagle_root + 'FLOWS'             ],
                      'utps': ['Unit Trust Prices'                      , eagle_root + 'UTPRICES'          ], 
                      'fnav': ['Fund Net Asset Value'                   , eagle_root + 'NetAsset'          ],
                      'tcrf': ['Trades Cross Reference'                 , eagle_root + 'TRADES%20REFERENCE'],
                      'cact': ['Cash Activity Details'                  , eagle_root + 'CSHACTIVITY'       ],
                     }

In [5]:
# utility function to save downloaded file with reporting date
def dater(folder_path, fund_name, dte):
    start_time = time.time()
    
    #from pathlib import Path
    import os
    fls     = os.listdir(folder_path)
    a       = max([os.path.abspath(os.path.join(folder_path, fl)) for fl in fls if r28N in fl], key = os.path.getmtime)
    z       = 'xlsx' if a[len(a) - 3:] == 'lsx' else 'xls'
    wb      = excel.Workbooks.Open(a)
    wb.Worksheets('Reg 28 Report - Incl Effective ').Range("J1").Value = dte.strftime("%d%b%Y")
    wb.SaveAs(os.path.abspath(os.path.join(folder_path, f'{fund_name} lookthrough {dte.strftime("%d%b%Y")}.{z}')))
    wb.Close()
    
    print(f'Downloaded file found and saved in {timediff(start_time, time.time())}') # time to get file name

In [6]:
# function to get latest file with specified extension in a given folder and chnage that file's name
import glob, os
from pathlib import Path

def latest_file(folder_path = str(Path.home() / "Downloads"), file_type = 'csv', new_file_name = 'newt'):
#https://datatofish.com/latest-file-python/
    files = glob.glob(folder_path + r'\*' + file_type)
    try:
        # latest file name
        max_file  = max(files, key=os.path.getctime)
        # rename the latest created file in the folder  https://www.squash.io/how-to-rename-a-file-with-python/
        path_new = os.path.join(folder_path, new_file_name) + '.' + file_type
        if os.path.exists(path_new):
            os.remove(path_new)
        os.rename(max_file, path_new)

    except ValueError as ve:
        max_file = f'No .{file_type} files in the folder. Error: {str(ve)}.'
    print(f'{max_file} -> {path_new}')
    #return

In [7]:
# an Eagle report lookup function, given six parameters
import datetime, time

def osprey(rpt_type = 'r28i', funds = 'PABS', d_from = datetime.datetime.today(), d_to = datetime.datetime.today(), sfx = 'csv', al = 'qt', xe = 'dk'):
    start_time          = time.time()

    # (1) load libraries
    #from datetime import datetime, timedelta
    %run utilities.ipynb
    import os
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys
    from selenium.webdriver.support.select import Select
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import NoSuchElementException
    # https://stackoverflow.com/questions/38022658/selenium-python-handling-no-such-element-exception
    
    # (2) set paths to the driver, urls, and to the report parameters
    import os
    os.environ["PATH"] = r'C:/SeleniumDrivers' # + os.pathsep + os.getenv("PATH")
    # https://stackoverflow.com/questions/61213005/modify-beginning-of-path-variable-with-os-environ-in-python
    pth                = r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm' # user_defined variables stored here
    url_default        = r'https://eagleportal.prescient.co.za/Default.aspx'

    # (3) prepare the report variables for fnds_, month_year_f, dayf, month_year_t, dayt,report_link
    fnds_        = funds
    month_year_f = f'{d_from:%B}, {d_from:%Y}' # e.g., 'January, 2023'
    dayf         = f'{d_from:%#d}'             # e.g., '03', i.e., report day with a leading zero f'{d_from:%d}'
    month_year_t = f'{d_to:%B},   {d_to:%Y}'   # e.g., 'January, 2023'
    dayt         = f'{d_to:%#d}'               # e.g., '03', i.e., report day with a leading zero f'{d_from:%d}'
    report_link  = report_types_dict[rpt_type][1]
    t = "0" if sfx == 'csv' else "4" # DXI4(0) for .xls(.csv)
    
    # (4) assign the browser driver
    from selenium import webdriver
    driver = webdriver.Firefox()
    
    # (5) open the browser on the Eagle web page
    driver.get(url_default)          # default page
    wait = WebDriverWait(driver, 10) # https://selenium-python.readthedocs.io/waits.html, max wait for elements to appear
    
    # (6) login
    driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_UserName'   ).send_keys(al)
    driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_Password'   ).send_keys(xe)
    driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_LoginButton').click()

    # (7) having logged in, open the reporting page (NEEDS report type link)
    driver.get(report_link)             # a hyperlink for the reporting page selected in the function osprey()
    
    # (8) switch to the query page
    driver.find_element(By.CSS_SELECTOR, '#ModifyLinkLabel').click()
    
    # (9) update the FROM calendar with month_year_f and dayf
    driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_B-1"]'      ).click() # date dropdown
    driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_NMC"]').click() # month advance
    driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_NMC"]').click() # month advance
    lmonth_selector = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 
                                         'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_PMC"]')))       # month regress
    while driver.find_element(By.XPATH,'//td[@id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_TC"]').text != month_year_f:
        lmonth_selector.click()
    day_selector = driver.find_element(By.XPATH,f'//td[@class="dxeCalendarDay"][text()={dayf}] | \
    //td[@class="dxeCalendarDay dxeCalendarWeekend"][text()={dayf}]')
    day_selector.click()

    try:
        #https://stackoverflow.com/questions/38022658/selenium-python-handling-no-such-element-exception
        # (9A) update the TO calendar with month_year_t and dayt        
        driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_B-1"]'      ).click() # date dropdown
        driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_DDD_C_NMC"]').click() # month advance
        driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_DDD_C_NMC"]').click() # month advance
        driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_DDD_C_NMC"]').click() # month advance  
        rmonth_selector = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 
                                             'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_DDD_C_PMC"]')))       # month regress
        while driver.find_element(By.XPATH,'//td[@id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_DDD_C_TC"]').text != month_year_t:
            rmonth_selector.click()
        day_selector = driver.find_element(By.XPATH,f'//td[@class="dxeCalendarDay"][text()={dayt}] | \
        //td[@class="dxeCalendarDay dxeCalendarWeekend"][text()={dayt}]')
        day_selector.click()
    except NoSuchElementException:
        pass
            
    # (10) get the web element for the FUND LIST and assign values to it
    fund_selector  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedIds"]')
    driver.execute_script(f'arguments[0].value = "{fnds_}";', fund_selector)
    
    # (11) click the table header where "Entity ID" resides
    WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 
                                            'table[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedItemsGrid_DXHeaderTable"]'))).click()
    time.sleep(5) # arbitrary 5 second wait

    # (12) get the web element of the 'Submit' button and then click it
    submit_button  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_RunBtn"]')
    submit_button.click()

    # (13) Wait for and then click the export button and then the xls download button
    #https://stackoverflow.com/questions/56085152/selenium-python-error-element-could-not-be-scrolled-into-view
    WebDriverWait(driver, 1000).until(EC.element_to_be_clickable((By.CSS_SELECTOR,   'a[id="DistrBtn"]'          ))).click()
    #t = "0" if sfx == 'csv' else "4" # DXI4(0) for .xls(.csv)
    WebDriverWait(driver, 1000).until(EC.element_to_be_clickable((By.CSS_SELECTOR, f'td[id="ExportMnu_DXI{t}_T"]'))).click()
    
    time.sleep(5) # wait for 5 seconds after the data downloads
    
    print(f'Downloading and then saving the {rpt_type} report in {sfx} format for {len(fnds_.split(","))} \
    funds: {timediff(start_time, time.time())}', '\n')

    # (14) having downloaded the requested report, close the web driver
    driver.quit()
    #print(f'Roundtrip time for getting holdings and derivative data: {timediff(start_time_overlord, time.time())}', '\n')

    # find the latest downloaded file and rename it
        # set the input variables for the latest file
    folder_path   = str(Path.home() / "Downloads")
    file_type     = sfx
    new_file_name = f'{rpt_type.upper()} ({len(fnds_.split(","))}) {d_from.strftime("%d%b%Y")}'
    
        # run the file name change function
    latest_file(folder_path, file_type, new_file_name) # gets the latest file of that type in the given folder and renames it to new_file_name
    
    print(f'  Roundtrip time to run the report lookup function: {timediff(start_time, time.time())}', '\n')

In [8]:
# convert .ipynb to .py         - https://stackoverflow.com/questions/17077494/how-do-i-convert-a-ipython-notebook-into-a-python-file-via-commandline
# constants from other notebook - https://stackoverflow.com/questions/6343330/importing-a-long-list-of-constants-to-a-python-file
# !jupyter nbconvert --to script utilities.ipynb     OR      %run utilities.ipynb